## TASK 3 Large-Scale ETL and Analysis in Spark ##

In [1]:
#!pip install pyspark==3.5.1

In [2]:
import os
import sys
import socket
import pyspark
from pyspark.sql import SparkSession

# networkk ---
# dont use socket.gethostname() because it returns 127.0.0.1 on Macs.
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.connect(("8.8.8.8", 80))
    local_ip = s.getsockname()[0]
    s.close()
except Exception:
    local_ip = "127.0.0.1"

print(f"PC IP: {local_ip}")
print(f"Workers will connect to: {local_ip}:20002")

os.environ['PYSPARK_PYTHON'] = 'python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# --- DATA PATHS ---
DATA_DIR = "DBahn-berlin" 

# The path inside the Docker container (as defined in the YAML file)
docker_base_path = "file:///opt/spark-data/DBahn-berlin"

   > PC IP: 10.18.77.124
   > Workers will connect to: 10.18.77.124:20002


In [3]:
spark = SparkSession.builder \
    .appName("spark_etl_3") \
    .master("spark://localhost:7077") \
    .config("spark.driver.host", local_ip) \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.port", "20002") \
    .config("spark.blockManager.port", "20003") \
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.18.0") \
    .config("spark.driver.memory", "5g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.memoryOverhead", "1g") \
    .config("spark.executor.instances", "1") \
    .config("spark.executor.cores", "2") \
    .config("spark.cores.max", "2") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.memory.fraction", "0.75") \
    .config("spark.memory.storageFraction", "0.25") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "1g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.initialPartitionNum", "16") \
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.sql.files.openCostInBytes", "4194304") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760") \
    .config("spark.network.timeout", "600s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.rpc.askTimeout", "600s") \
    .config("spark.rpc.lookupTimeout", "600s") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.parquet.writeLegacyFormat", "false") \
    .config("spark.sql.parquet.mergeSchema", "false") \
    .config("spark.sql.parquet.filterPushdown", "true") \
    .config("spark.speculation", "false") \
    .config("spark.task.maxFailures", "4") \
    .config("spark.stage.maxConsecutiveAttempts", "4") \
    .config("spark.sql.sources.parallelPartitionDiscovery.threshold", "0") \
    .getOrCreate()

print("Spark Session created")

25/12/31 16:10:25 WARN Utils: Your hostname, Aycas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.18.77.124 instead (on interface en0)
25/12/31 16:10:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/aycaaksoy/.ivy2/cache
The jars for the packages stored in: /Users/aycaaksoy/.ivy2/jars
com.databricks#spark-xml_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f1254b5f-ff05-44bf-b11c-d49290ac73c8;1.0
	confs: [default]
	found com.databricks#spark-xml_2.12;0.18.0 in central
	found commons-io#commons-io;2.11.0 in local-m2-cache
	found org.glassfish.jaxb#txw2;3.0.2 in central
	found org.apache.ws.xmlschema#xmlschema-core;2.3.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.9.0 in central
:: resolution report :: resolve 79ms :: artifacts dl 4ms
	:: modules in use:
	com.databricks#spark-xml_2.12;0.18.0 from central in [default]
	commons-io#commons-

:: loading settings :: url = jar:file:/Users/aycaaksoy/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


25/12/31 16:10:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark Session created


In [4]:
from datetime import datetime
import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import *
from pyspark.sql.functions import col, coalesce, unix_timestamp, to_date, broadcast, lit, avg, round, count, hour, desc, input_file_name, regexp_extract, substring_index, to_timestamp, when, explode, to_timestamp, regexp_replace,lower, trim
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, DoubleType, TimestampType, BooleanType


### TASK 3.1  Spark ETL job ### 

In [5]:
# Schema for Station Data
station_schema = StructType([
    StructField("result", ArrayType(StructType([
        StructField("name", StringType(), True),
        StructField("category", LongType(), True),
        StructField("evaNumbers", ArrayType(StructType([
            StructField("number", LongType(), True),
            StructField("isMain", BooleanType(), True),
            StructField("geographicCoordinates", StructType([
                StructField("coordinates", ArrayType(DoubleType()), True)
            ]), True)
        ])), True)
    ])), True)
])

In [6]:
df_stations = spark.read.schema(station_schema) \
    .option("multiLine", True) \
    .json(f"{docker_base_path}/station_data.json") \
    .selectExpr("explode(result) as res") \
    .select(
        col("res.name").alias("meta_station_name"),
        col("res.category").alias("station_category"),
        explode(col("res.evaNumbers")).alias("eva_data")
    ) \
    .filter(col("eva_data.isMain") == True) \
    .select(
        col("meta_station_name"),
        col("station_category"),
        col("eva_data.number").alias("station_eva_id"),
        col("eva_data.geographicCoordinates.coordinates")[0].alias("longitude"),
        col("eva_data.geographicCoordinates.coordinates")[1].alias("latitude")
    )

In [7]:
# function for  both tables to use the exact same normalization rules.
    # 1. Lowercase
    # 2. Remove "Berlin" context-free (handles prefix "Berlin-" and infix "Berlin")
    # 3. Remove all non-alphanumeric chars (spaces, brackets, dashes)
def create_join_key(column_name):
    c = lower(col(column_name))
    c = regexp_replace(c, "berlin", "") 
    c = regexp_replace(c, r"[^a-z0-9äöüß]", "") 
    return c


# prepare station data
df_stations_w_joink = df_stations.withColumn(
    "join_key", 
    create_join_key("meta_station_name")
)
# small table caching is useful
df_stations_w_joink.cache()
print(f"Stations loaded: {df_stations_w_joink.count()}")  

[Stage 4:>                                                          (0 + 1) / 1]

Stations loaded: 133


In [8]:
total_rows = df_stations_w_joink.count()
df_stations_w_joink.show(n=total_rows, truncate=False)

+----------------------------------+----------------+--------------+----------+----------+-----------------------------+
|meta_station_name                 |station_category|station_eva_id|longitude |latitude  |join_key                     |
+----------------------------------+----------------+--------------+----------+----------+-----------------------------+
|Ahrensfelde                       |4               |8011003       |13.565154 |52.571375 |ahrensfelde                  |
|Albrechtshof                      |5               |8080040       |13.128917 |52.549396 |albrechtshof                 |
|Alexanderplatz                    |3               |8011155       |13.410961 |52.521481 |alexanderplatz               |
|Altglienicke                      |4               |8089054       |13.558753 |52.407299 |altglienicke                 |
|Attilastraße                      |4               |8089003       |13.360951 |52.447681 |attilastraße                 |
|Baumschulenweg                 

In [9]:
# planned timetables schema define
timetable_schema = StructType([
    StructField("_station", StringType(), True),
    StructField("s", ArrayType(StructType([
        StructField("_id", StringType(), True),
        StructField("ar", StructType([
            StructField("_pt", StringType(), True),
            StructField("_pp", StringType(), True),
            StructField("_l", StringType(), True)
        ]), True),
        StructField("dp", StructType([
            StructField("_pt", StringType(), True),
            StructField("_l", StringType(), True)
        ]), True),
        StructField("tl", StructType([
            StructField("_c", StringType(), True),
            StructField("_n", StringType(), True),
            StructField("_o", StringType(), True)
        ]), True)
    ]), True))
])

In [10]:
# read timetables xml 
df_planned = spark.read.format("xml") \
    .option("rowTag", "timetable") \
    .schema(timetable_schema) \
    .load(f"{docker_base_path}/timetables/*/*.xml") \
    .withColumn("source_file", input_file_name()) \
    .select(
        col("source_file"),
        col("_station").alias("xml_station_name"),
        explode(col("s")).alias("stop")
    )

In [11]:
# data cleaning for planned timetables

df_planned_fixed = df_planned.withColumn("clean_name", trim(col("xml_station_name")))

# some unmatched station names are normalised
df_planned_fixed = df_planned_fixed.withColumn("clean_name", 
    when(col("clean_name").like("%Yorckstr.(S1)%"), "Yorckstraße (Großgörschenstraße)")
    .when(col("clean_name").like("%Yorckstr.(S2)%"), "Yorckstraße")
    .when(col("clean_name").like("%Berlin-Schulzendorf%"), "Schulzendorf (b Tegel)")
    .when(col("clean_name").like("%Berlin-Schöneweide%"), "Berlin-Schöneweide Pbf") 
    .otherwise(col("clean_name"))
)

# Regex Cleaning
# 1. Remove noise
df_planned_fixed = df_planned_fixed.withColumn("clean_name", regexp_replace("clean_name", r"\s*\(S\)", ""))
df_planned_fixed = df_planned_fixed.withColumn("clean_name", regexp_replace("clean_name", r"\s*\(Witzleben\)", ""))

# 2. Fix 'Betriebsbf' -> 'Betriebsbahnhof' (Must run BEFORE generic Bf check)
df_planned_fixed = df_planned_fixed.withColumn("clean_name", regexp_replace("clean_name", r"(?i)Betriebsbf", "Betriebsbahnhof"))

# 3. Fix 'Hbf' -> 'Hauptbahnhof'
df_planned_fixed = df_planned_fixed.withColumn("clean_name", regexp_replace("clean_name", r"(?i)Hbf", "Hauptbahnhof"))

# 4. Fix 'Bf' -> 'Bahnhof'
# The regex \bBf\b matches "Bf" only when it is a distinct word (e.g., "Anhalter Bf").
df_planned_fixed = df_planned_fixed.withColumn("clean_name", regexp_replace("clean_name", r"(?i)\bBf\b", "Bahnhof"))

# 5. Fix 'Str' -> 'Straße' (Handles Str. and Str at end of line)
df_planned_fixed = df_planned_fixed.withColumn("clean_name", regexp_replace("clean_name", r"(?i)Str\.?(\s|$)", "Straße "))

# Create Join Key
df_planned_fixed = df_planned_fixed.withColumn(
    "join_key", 
    create_join_key("clean_name")
)

# join
df_planned_joined = df_planned_fixed.join(
    df_stations_w_joink,
    on="join_key",
    how="left"
)

In [12]:
df_planned_final = df_planned_joined.select(
    col("stop._id").alias("stop_id"),            
    col("station_eva_id"),
    col("meta_station_name").alias("station_name"),
    regexp_extract("source_file", r".*/(\d+)/.*", 1).alias("batch_id"),
    col("stop.ar._pt").alias("planned_arrival_str"),
    col("stop.dp._pt").alias("planned_departure_str"),
    col("stop.ar._pp").alias("planned_platform"),
    col("stop.tl._c").alias("train_category"),
    col("stop.tl._n").alias("train_number"),
    col("stop.tl._o").alias("train_owner"),
    coalesce(col("stop.ar._l"), col("stop.dp._l")).alias("line_name"),
    col("latitude"),
    col("longitude")
)

In [13]:
# null check station_eva_id
print("--- Failed joins ---")
df_planned_joined \
    .filter(col("station_eva_id").isNull()) \
    .select("xml_station_name") \
    .distinct() \
    .show(133,truncate=False)


--- Failed joins ---


[Stage 10:===============================================>(135635 + 2) / 135692]

+----------------+
|xml_station_name|
+----------------+
+----------------+



[Stage 10:===============================================>(135690 + 2) / 135692]

In [14]:
# define changed timetables xml schema
changes_schema = StructType([
    StructField("_id", StringType(), True),
    StructField("ar", StructType([
        StructField("_ct", StringType(), True),
        StructField("_l", StringType(), True),
        StructField("_clt", StringType(), True)
    ]), True),
    StructField("dp", StructType([
        StructField("_ct", StringType(), True),
        StructField("_clt", StringType(), True)
    ]), True)
])

# read changed timetables xml
df_changes = spark.read.format("xml") \
    .option("rowTag", "s") \
    .schema(changes_schema) \
    .load(f"{docker_base_path}/timetable_changes/*/*.xml") \
    .withColumn("batch_id", regexp_extract(input_file_name(), r".*/(\d+)/.*", 1)) \
    .select(
        col("_id").alias("stop_id"),
        col("batch_id"),
        col("ar._ct").alias("actual_arrival_str"),
        col("dp._ct").alias("actual_departure_str"),
        col("ar._l").alias("changed_platform"),
        (col("ar._clt").isNotNull() | col("dp._clt").isNotNull()).alias("is_cancelled")
    )

# merge all
df_final = df_planned_final.join(df_changes, ["stop_id", "batch_id"], "left") \
    .withColumn("planned_arr_ts", to_timestamp(col("planned_arrival_str"), "yyMMddHHmm")) \
    .withColumn("planned_dep_ts", to_timestamp(col("planned_departure_str"), "yyMMddHHmm")) \
    .withColumn("actual_arr_ts", to_timestamp(col("actual_arrival_str"), "yyMMddHHmm")) \
    .withColumn("actual_dep_ts", to_timestamp(col("actual_departure_str"), "yyMMddHHmm")) \
    .withColumn("final_arr_ts", coalesce(col("actual_arr_ts"), col("planned_arr_ts"))) \
    .withColumn("final_dep_ts", coalesce(col("actual_dep_ts"), col("planned_dep_ts"))) \
    .withColumn("arrival_delay_min", (unix_timestamp(col("final_arr_ts")) - unix_timestamp(col("planned_arr_ts"))) / 60) \
    .withColumn("departure_delay_min", (unix_timestamp(col("final_dep_ts")) - unix_timestamp(col("planned_dep_ts"))) / 60) \
    .withColumn("final_platform", coalesce(col("changed_platform"), col("planned_platform"))) \
    .withColumn("is_cancelled", coalesce(col("is_cancelled"), lit(False))) \
    .withColumn("event_date", to_date(coalesce(col("planned_dep_ts"), col("planned_arr_ts"))))


In [15]:

# write to parquet

# output path inside Docker
output_path = "file:///opt/spark-data/train_movements_final.parquet"

# Calculate optimal coalesce number for optimization
num_dates = df_final.select("event_date").distinct().count()
print(f"Number of unique dates: {num_dates}")

# Strategy: 2-4 files per date partition for parallel write
coalesce_factor = max(8, num_dates * 2)

df_final \
    .repartition(col("event_date")) \
    .coalesce(coalesce_factor) \
    .write \
    .partitionBy("event_date") \
    .mode("overwrite") \
    .option("maxRecordsPerFile", "150000") \
    .option("compression", "snappy") \
    .parquet(output_path)


print("ETL Completed Successfully!")

25/12/31 16:25:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Number of unique dates: 44


[Stage 26:===================================================>    (12 + 1) / 13]

ETL Completed Successfully!


### TASK 3.2 ### 

In [18]:
# load the parquet we created

df = spark.read.parquet(output_path)

test_station = "Berlin-Nordbahnhof"

print(f"Computing Average Daily Delay for: {test_station}")

# Filter for the specific station
station_df = df.filter(col("station_name") == test_station)


daily_delay = station_df.groupBy("event_date") \
    .agg(count("*").alias("total_trains"),
        round(avg("arrival_delay_min"), 2).alias("avg_arrival_delay_min"),
        round(avg("departure_delay_min"), 2).alias("avg_departure_delay_min")) \
        .orderBy("event_date")


print(f"--- Results for {test_station} ---")
daily_delay.show(20, truncate=False)

# Extra - Calculate the global average for the entire period
overall_avg = station_df.select(
    round(avg("arrival_delay_min"), 2).alias("period_avg_arrival_delay"),
    round(avg("departure_delay_min"), 2).alias("period_avg_departure_delay")).collect()[0]

print(f"Overall Period Average for {test_station}:")
print(f"Arrival Delay:   {overall_avg['period_avg_arrival_delay']} min")
print(f"Departure Delay: {overall_avg['period_avg_departure_delay']} min")

Computing Average Daily Delay for: Berlin-Nordbahnhof
--- Results for Berlin-Nordbahnhof ---
+----------+------------+---------------------+-----------------------+
|event_date|total_trains|avg_arrival_delay_min|avg_departure_delay_min|
+----------+------------+---------------------+-----------------------+
|2025-09-02|407         |0.01                 |0.01                   |
|2025-09-03|702         |0.07                 |0.06                   |
|2025-09-04|666         |0.12                 |0.1                    |
|2025-09-05|708         |0.07                 |0.06                   |
|2025-09-06|601         |0.04                 |0.03                   |
|2025-09-07|566         |0.04                 |0.04                   |
|2025-09-08|704         |0.44                 |0.43                   |
|2025-09-09|704         |0.23                 |0.21                   |
|2025-09-10|705         |0.3                  |0.31                   |
|2025-09-11|704         |0.22              

### TASK 3.3 ### 

In [17]:
# Avg number of departures during peak hours (07:00-09:00 and 17:00-19:00)

# Filter Peak Hours Only
# filter out cancelled trains 
df_peak = df.filter(
    hour(col("planned_dep_ts")).isin(7, 8, 17, 18) &
    (col("is_cancelled") == False)
)

# aggregate Count departures per Station per Day
daily_peak_counts = df_peak.groupBy("station_name", "event_date") \
    .agg(count("*").alias("daily_departures"))

# average daily counts over the entire period
avg_peak_departures = daily_peak_counts.groupBy("station_name") \
    .agg(round(avg("daily_departures"), 1).alias("avg_peak_departures")) \
    .orderBy(desc("avg_peak_departures"))

avg_peak_departures.show(133, truncate=False)

[Stage 35:====================================================>   (14 + 1) / 15]

+----------------------------------+-------------------+
|station_name                      |avg_peak_departures|
+----------------------------------+-------------------+
|Berlin-Westkreuz                  |223.3              |
|Warschauer Straße                 |212.3              |
|Schöneberg                        |171.6              |
|Bornholmer Straße                 |168.7              |
|Berlin Anhalter Bahnhof           |158.0              |
|Nöldnerplatz                      |153.9              |
|Friedrichsfelde Ost               |152.1              |
|Savignyplatz                      |150.5              |
|Jannowitzbrücke                   |150.4              |
|Bellevue                          |150.3              |
|Hackescher Markt                  |150.2              |
|Tiergarten                        |150.2              |
|Treptower Park                    |146.8              |
|Berlin-Neukölln                   |145.8              |
|Baumschulenweg                